# Benchmark Raspberry Pi — YOLOv8n NCNN
Versión optimizada para Raspberry Pi 4 usando **NCNN** como backend de inferencia.  
Misma estructura de métricas que `ScriptRaspberry.ipynb` (versión PC/GPU) para comparar directamente.

## Paso 0 — Exportar modelo a NCNN (una sola vez)
Correr esta celda **una única vez** (en PC o en la RPi). Genera `models/yolov8n_ncnn_model/`.  
Si la carpeta ya existe, salteá esta celda.

In [ ]:
import os
from ultralytics import YOLO

ruta_proyecto = os.path.abspath(os.path.join(os.getcwd(), '..'))
ruta_pt       = os.path.join(ruta_proyecto, 'models', 'yolov8n.pt')
ruta_ncnn     = os.path.join(ruta_proyecto, 'models', 'yolov8n_ncnn_model')

if os.path.isdir(ruta_ncnn):
    print(f'Modelo NCNN ya existe en: {ruta_ncnn}')
else:
    model_pt = YOLO(ruta_pt)
    model_pt.export(format='ncnn')
    print(f'Exportacion completada -> {ruta_ncnn}')

## Benchmark

In [ ]:
import sys
import os
import time
import cv2
import numpy as np
import psutil
import platform

ruta_proyecto = os.path.abspath(os.path.join(os.getcwd(), '..'))
os.chdir(ruta_proyecto)
sys.path.insert(0, os.path.join(ruta_proyecto, 'src'))

from detection.yolo_detection import YoloDetection
from distanceEstimation.Distance_Estimation import DistanceEstimation

print(f'Plataforma : {platform.machine()} --- {platform.system()} {platform.release()}')
print(f'Python     : {platform.python_version()}')
print(f'Directorio : {os.getcwd()}')

In [ ]:
# Cargar modelo NCNN y video
rutaModelo = os.path.join(ruta_proyecto, 'models', 'yolov8n_ncnn_model')
rutaVideo  = os.path.join(ruta_proyecto, 'data', 'samples', 'Val.mp4')

detector = YoloDetection(rutaModelo)
detector.model.overrides['verbose'] = False

cap = cv2.VideoCapture(rutaVideo)
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
fps_video    = cap.get(cv2.CAP_PROP_FPS)
print(f'Video: {total_frames} frames a {fps_video:.1f} FPS')

N_WARMUP = 20
for _ in range(N_WARMUP):
    ret, frame_warmup = cap.read()
    if ret:
        detector.detectAndParse(frame_warmup)
cap.set(cv2.CAP_PROP_POS_FRAMES, 0)
print(f'Warm-up completado ({N_WARMUP} frames)')

In [ ]:
# Loop de benchmark
tiempos_deteccion = []
tiempos_distancia = []
tiempos_umbral    = []
tiempos_total     = []
uso_cpu           = []
uso_ram_mb        = []

proceso = psutil.Process(os.getpid())
cap.set(cv2.CAP_PROP_POS_FRAMES, 0)

while True:
    ret, frame = cap.read()
    if not ret:
        break

    t_inicio = time.perf_counter()

    t1 = time.perf_counter()
    detecciones = detector.detectAndParse(frame)
    t2 = time.perf_counter()

    z_ref, obj_min, distancias_vec = None, None, []
    try:
        z_ref, obj_min, distancias_vec = DistanceEstimation.distanciasIntervehiculares(detecciones)
    except Exception:
        pass
    t3 = time.perf_counter()

    try:
        if z_ref is not None:
            DistanceEstimation.clasificacionDeDistancia(z_ref)
        for d_AB, _ in distancias_vec:
            DistanceEstimation.clasificacionDeDistancia(d_AB)
    except Exception:
        pass
    t4 = time.perf_counter()

    tiempos_deteccion.append(t2 - t1)
    tiempos_distancia.append(t3 - t2)
    tiempos_umbral.append(t4 - t3)
    tiempos_total.append(t4 - t_inicio)
    uso_cpu.append(psutil.cpu_percent(interval=None))
    uso_ram_mb.append(proceso.memory_info().rss / 1024**2)

cap.release()
print(f'Frames procesados: {len(tiempos_total)}')

In [ ]:
# Resultados
PLATAFORMA = 'Raspberry Pi'

def resumen(nombre, datos_s):
    datos_ms = np.array(datos_s) * 1000
    print(f'  {nombre}')
    print(f'    Media   : {datos_ms.mean():.3f} ms')
    print(f'    Mediana : {np.median(datos_ms):.3f} ms')
    print(f'    P95     : {np.percentile(datos_ms, 95):.3f} ms')
    print(f'    P99     : {np.percentile(datos_ms, 99):.3f} ms')
    print(f'    Min     : {datos_ms.min():.3f} ms')
    print(f'    Max     : {datos_ms.max():.3f} ms')

n = len(tiempos_total)
fps_real = 1 / np.mean(tiempos_total)

print(f'========== BENCHMARK --- {PLATAFORMA} ==========')
print(f'Frames procesados : {n}')
print()
resumen('1. Deteccion YOLOv8n-NCNN',     tiempos_deteccion)
print()
resumen('2. Estimacion de distancias',    tiempos_distancia)
print()
resumen('3. Clasificacion por umbrales',  tiempos_umbral)
print()
resumen('Pipeline completo (1+2+3)',      tiempos_total)
print(f'    FPS estimados : {fps_real:.2f}')
print()
print(f'  CPU / RAM')
print(f'    CPU media : {np.mean(uso_cpu):.1f}%')
print(f'    RAM media : {np.mean(uso_ram_mb):.1f} MB')
print(f'    RAM pico  : {np.max(uso_ram_mb):.1f} MB')